# Finetune with Contrastive-Learned Text Encoder


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DIR"]        = "/kaggle/tmp"
os.environ["WANDB_CACHE_DIR"]  = "/kaggle/tmp"
os.environ["WANDB_CONFIG_DIR"] = "/kaggle/tmp"

In [ ]:
import os, sys

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = False
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    pass

WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
print(f"Runtime: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Other'}")
print(f"Working dir: {WORK_DIR}")

os.system("pip install -q selfies wandb pyarrow")
os.system("pip install -q 'rdkit>=2023.3'")
os.system("pip install -q 'transformers>=4.40' 'sentence-transformers>=2.6'")
print("Dependencies ready.")

In [ ]:
import subprocess, shutil

GIT_BRANCH = "prompt-condition"
REPO_NAME  = "morpheus"
REPO_PATH  = os.path.join(WORK_DIR, REPO_NAME)

!git clone -b {GIT_BRANCH} https://github.com/vivaikmalik/morpheus.git {REPO_PATH}

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)

In [ ]:
import wandb
os.environ.pop("WANDB_MODE", None)

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
else:
    wandb.login()

In [ ]:
import zipfile

dataset_url = "https://huggingface.co/datasets/zjunlp/Mol-Instructions/resolve/main/data/Molecule-oriented_Instructions.zip"
zip_file    = "Molecule-oriented_Instructions.zip"
extract_dir = "./data_mol_instruct"

if not os.path.exists(extract_dir):
    print("Downloading dataset...")
    os.system(f"wget -q {dataset_url} -O {zip_file}")
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(extract_dir)
    print("Extraction complete.")
else:
    print("Dataset already present.")

In [ ]:
import pandas as pd
import json

json_path = os.path.join(extract_dir, "Molecule-oriented_Instructions",
                         "description_guided_molecule_design.json")
with open(json_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)

def format_prompt(row):
    instruction = row.get('instruction', '')
    inp = row.get('input', '')
    return f"{instruction}\nInput: {inp}" if inp else instruction

df['prompt']   = df.apply(format_prompt, axis=1)
df['response'] = df['output']
df['split']    = df['metadata'].apply(lambda m: m['split'])

train_df_ = df[df['split'] == 'train'][['prompt', 'response']]
train_df  = train_df_.sample(frac=0.9, random_state=42)
val_df    = train_df_.drop(train_df.index)
test_df   = df[df['split'] == 'test'][['prompt', 'response']]

train_csv_path = os.path.join(WORK_DIR, "train.csv")
val_csv_path   = os.path.join(WORK_DIR, "val.csv")
train_df.to_csv(train_csv_path, index=False)
val_df.to_csv(val_csv_path,   index=False)

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

In [ ]:
DATASET_CSV = os.path.join(WORK_DIR, "train.csv")
assert os.path.exists(DATASET_CSV)
df_check = pd.read_csv(DATASET_CSV, nrows=3)
assert "prompt"   in df_check.columns
assert "response" in df_check.columns
print(f"Dataset OK — {len(pd.read_csv(DATASET_CSV)):,} rows")
print(df_check[["prompt", "response"]].to_string())

In [ ]:
from pathlib import Path

shutil.copy(train_csv_path, os.path.join(REPO_PATH, "train.csv"))
shutil.copy(val_csv_path,   os.path.join(REPO_PATH, "val.csv"))

checkpoint_dir     = os.path.join(REPO_PATH, "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)
PRETRAINED_WEIGHTS = Path(checkpoint_dir) / "best_model.pt"
if PRETRAINED_WEIGHTS.exists():
    print(f"Pre-trained weights found: {PRETRAINED_WEIGHTS}")
else:
    print("WARNING: best_model.pt not found — will train from scratch.")

In [ ]:
from finetune.train_text_condition import train, CONFIG

CONFIG["data_path"]     = "train.csv"
CONFIG["val_data_path"] = "val.csv"
CONFIG["num_workers"]   = 8
CONFIG["batch_size"]    = 512
CONFIG["log_every"]     = 1
CONFIG["num_epochs"]    = 25
CONFIG["wandb_project"] = "morpheus-diffusion-finetune-contrastive"

# Set one of these to load the encoder
CONFIG["contrastive_artifact"]  = None  # Wnadb artifact name
CONFIG["contrastive_ckpt_path"] = None  # Local path

print("Final CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k:30s} = {v}")

print("\nStarting training...")
train()

In [ ]:
FINETUNED = os.path.join(REPO_PATH, "checkpoints", "best_finetuned_model.pt")

if os.path.exists(FINETUNED):
    dest = os.path.join(WORK_DIR, "best_finetuned_model_contrastive.pt")
    shutil.copy(FINETUNED, dest)
    print(f"Checkpoint saved to {dest}")
else:
    print("No checkpoint found — training may not have completed a validation step yet.")

# Interactive Inference

In [ ]:
import torch
import selfies as sf
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display
from transformers import AutoTokenizer

from tokenizer.chemicalTokenizer import ChemicalTokenizer
from model.molecularDiffusionModel import MolecularDiffusionModel
from finetune.train_text_condition import CONFIG

PROMPTS = [
    "A highly soluble molecule with a benzene ring.",
    "A small drug-like fragment with low molecular weight.",
    "A molecule containing a fluorine atom.",
    "A complex polycyclic aromatic compound.",
]

CFG_SCALE   = 3.0
NUM_STEPS   = 32
TEMPERATURE = 1.0

INFERENCE_CHECKPOINT = os.path.join(WORK_DIR, "best_finetuned_model_contrastive.pt")

device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer    = ChemicalTokenizer(os.path.join(REPO_PATH, "chemical_tokenizer.json"))
hf_tokenizer = AutoTokenizer.from_pretrained(CONFIG["text_model"])

model = MolecularDiffusionModel(
    vocab_size      = CONFIG["vocab_size"],
    hidden_size     = CONFIG["hidden_size"],
    num_heads       = CONFIG["num_heads"],
    ffn_dim         = CONFIG["ffn_dim"],
    num_layers      = CONFIG["num_layers"],
    max_length      = CONFIG["max_length"],
    pad_token_id    = tokenizer.pad_token_id,
    text_model_name = CONFIG["text_model"],
    dropout         = 0.0,
).to(device)

assert os.path.exists(INFERENCE_CHECKPOINT), \
    f"Checkpoint not found: {INFERENCE_CHECKPOINT}"
ckpt = torch.load(INFERENCE_CHECKPOINT, map_location=device)
model.load_state_dict(ckpt["model"], strict=False)
model.eval()
print(f"Loaded: {INFERENCE_CHECKPOINT}\n")

max_length = CONFIG["max_length"]
num_mols   = len(PROMPTS)

text_inputs       = hf_tokenizer(PROMPTS, padding=True, truncation=True, return_tensors="pt").to(device)
text_padding_mask = (text_inputs["attention_mask"] == 0)

with torch.no_grad():
    text_embeds = model.get_text_embeddings(text_inputs["input_ids"], text_inputs["attention_mask"])
    null_embeds = model.null_token.expand(num_mols, text_embeds.size(1), -1)
    input_ids   = torch.full((num_mols, max_length), tokenizer.mask_token_id, device=device)
    t_vals      = torch.linspace(1.0, 0.0, NUM_STEPS, device=device)

    for step_idx, t_val in enumerate(t_vals):
        step_t        = t_val.repeat(num_mols).unsqueeze(-1)
        cond_logits   = model(input_ids, step_t, text_embeds,  text_padding_mask)
        uncond_logits = model(input_ids, step_t, null_embeds,  text_padding_mask)
        logits        = uncond_logits + CFG_SCALE * (cond_logits - uncond_logits)
        if step_idx < NUM_STEPS - 1:
            logits[:, :, tokenizer.mask_token_id] = float('-inf')
        probs   = torch.softmax(logits / TEMPERATURE, dim=-1)
        sampled = torch.distributions.Categorical(probs=probs).sample()
        conf    = torch.gather(probs, 2, sampled.unsqueeze(-1)).squeeze(-1)
        alpha_t = (torch.cos(t_val * torch.pi / 2) ** 2).item()
        n_mask  = int((1.0 - alpha_t) * max_length)
        if n_mask > 0 and step_idx < NUM_STEPS - 1:
            _, mask_idx = torch.topk(conf, n_mask, dim=-1, largest=False)
            sampled.scatter_(1, mask_idx, tokenizer.mask_token_id)
        input_ids = sampled

mols = []
for i in range(num_mols):
    ids = input_ids[i].cpu().tolist()
    if tokenizer.eos_token_id in ids:
        ids = ids[:ids.index(tokenizer.eos_token_id)]
    selfies_str = tokenizer.decode(ids)
    try:
        smiles = sf.decoder(selfies_str)
        mol    = Chem.MolFromSmiles(smiles)
        if mol:
            mols.append(mol)
            print(f"[{i}] Valid   {Chem.MolToSmiles(mol)}")
        else:
            mols.append(None)
            print(f"[{i}] Invalid SELFIES: {selfies_str}")
    except Exception as e:
        mols.append(None)
        print(f"[{i}] Error: {e}")
    print(f"     Prompt: {PROMPTS[i]}")

valid_mols = [m for m in mols if m is not None]
if valid_mols:
    display(Draw.MolsToGridImage(valid_mols, molsPerRow=2, subImgSize=(300, 300)))